# Stage 4.5 - Run 2b reality check (trajectory-post-proc coverage)

**Upload, Run All.** No training. Uses `MyDrive/pb_v4_upload.zip` (bundle) +
`MyDrive/ball_model_v4_run2.pt` (the Run-2b model). Runtime -> GPU.

Reports, per venue, **raw per-frame recall** vs **effective coverage** after the
Stage-4 trajectory post-processing (gap-fill <= 8 frames + velocity-outlier drop)
on the same held-out slices used in training. Big raw->effective lift = the misses
are short recoverable gaps; little lift = long motion-blur runs (needs more data).

In [ ]:
# Reality check: does trajectory post-processing lift court2/indoor to usable
# ball.parquet coverage? Runs the saved Run-2b model over the SAME per-venue
# held-out slices and reports RAW per-frame recall vs EFFECTIVE coverage after
# the Stage-4 trajectory post-proc (gap-fill <= MAX_GAP frames + outlier drop).
import os, json, zipfile, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
LOCAL = Path('/content/pb_v4')
if not LOCAL.exists():
    print('unzipping bundle...')
    with zipfile.ZipFile('/content/drive/MyDrive/pb_v4_upload.zip') as z:
        z.extractall(LOCAL)
REPO = LOCAL/'repo'; DATA = LOCAL/'data'
sys.path.insert(0, str(REPO))
try:
    import cv2
except Exception:
    os.system('pip -q install opencv-python-headless'); import cv2
import numpy as np, torch
cv2.setNumThreads(0)
from stages.track_ball._tracknet_model import TrackNet

PROC_H, PROC_W = 720, 1280
CONF, TOL_PROC = 0.30, 6           # detection threshold + correctness tol (proc px), matches training
SX = 3.0                           # 4K source / 1280 proc -> px scale (all clips 3840x2160)
MAX_GAP, OUTLIER = 8, 250.0        # Stage-4 track_ball_v4 constants (source px)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', dev)

def clip_items(folder):
    m = json.load(open(folder/'v4_manifest.json'))
    fd = folder/m.get('frames_dir', 'frames_720')
    return [(s, fd) for s in m['samples']]

def _fk(s):
    fr = s['frames']; return fr[len(fr)//2] if isinstance(fr,(list,tuple)) else fr

def temporal_split(items, val_frac=0.12):
    order = sorted(range(len(items)), key=lambda k: _fk(items[k][0]))
    n_val = max(1, int(round(len(items)*val_frac)))
    return [items[k] for k in order[-n_val:]]   # just the val slice

def rd(fd, i):
    im = cv2.imread(str(fd/f'{i}.jpg'))
    if im is None: return None
    if im.shape[:2] != (PROC_H, PROC_W): im = cv2.resize(im, (PROC_W, PROC_H))
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0

def postprocess(dets, frames):
    """Faithful copy of stages/track_ball/track_ball_v4.postprocess (source px)."""
    cf = sorted(dets.keys()); kept = set(cf)
    for i, f in enumerate(cf):
        x, y, _ = dets[f]; bad = []
        for j in (i-1, i+1):
            if 0 <= j < len(cf):
                g = cf[j]; px, py, _ = dets[g]
                bad.append(np.hypot(x-px, y-py) > OUTLIER*max(1, abs(f-g)))
        if bad and all(bad): kept.discard(f)
    conf = [f for f in cf if f in kept]; confset = set(conf)
    interp = {}
    for a, b in zip(conf, conf[1:]):
        gap = b-a
        if 1 < gap <= MAX_GAP:
            xa, ya, _ = dets[a]; xb, yb, _ = dets[b]
            for k in range(1, gap):
                t = k/gap; interp[a+k] = (xa+t*(xb-xa), ya+t*(yb-ya))
    return confset, interp

@torch.no_grad()
def eval_traj(model, items, name):
    order = sorted(range(len(items)), key=lambda k: _fk(items[k][0]))
    dets, label = {}, {}
    for k in order:
        s, fd = items[k]; f = _fk(s)
        frs = [rd(fd, i) for i in s['frames']]
        if any(fr is None for fr in frs): continue
        st = np.concatenate([fr.transpose(2,0,1) for fr in frs], 0)[None].astype(np.float32)
        with torch.cuda.amp.autocast(enabled=dev=='cuda'):
            hm = model(torch.from_numpy(st).to(dev))[0,0].float().cpu().numpy()
        iy, ix = np.unravel_index(int(hm.argmax()), hm.shape); c = float(hm[iy,ix])
        if c >= CONF: dets[f] = (ix*SX, iy*SX, c)
        label[f] = (s['x_proc']*SX, s['y_proc']*SX) if s['visible'] else None
    frames = sorted(label.keys())
    confset, interp = postprocess(dets, frames)
    vis = [f for f in frames if label[f] is not None]
    nv  = [f for f in frames if label[f] is None]
    tol = TOL_PROC*SX
    def near(f, pos): lx,ly = label[f]; return np.hypot(pos[0]-lx, pos[1]-ly) <= tol
    raw = sum(1 for f in vis if f in confset and near(f, dets[f][:2]))
    rec = sum(1 for f in vis if (f in confset and near(f, dets[f][:2])) or (f in interp and near(f, interp[f])))
    fp  = sum(1 for f in nv if f in confset or f in interp)
    # gap histogram between consecutive confirmed detections (fillable <=MAX_GAP vs not)
    cf = sorted(f for f in dets if f in confset)
    gaps = [b-a for a,b in zip(cf, cf[1:]) if b-a > 1]
    fill = sum(1 for g in gaps if g <= MAX_GAP); long = sum(1 for g in gaps if g > MAX_GAP)
    print(f"{name:7s} raw {raw/max(len(vis),1):.3f} -> EFFECTIVE {rec/max(len(vis),1):.3f}  "
          f"fp {fp/max(len(nv),1):.3f}  (vis {len(vis)}, notvis {len(nv)}, "
          f"gaps fillable<= {MAX_GAP}: {fill}, long: {long})")

ck = torch.load('/content/drive/MyDrive/ball_model_v4_run2.pt', map_location=dev)
model = TrackNet(in_channels=9, out_channels=1, input_shape=(PROC_H,PROC_W)).to(dev)
model.load_state_dict(ck['state_dict']); model.eval()
print('loaded run2 model; per-venue saved recall:', ck.get('per_venue'))
print('--- raw per-frame  vs  effective (after Stage-4 trajectory post-proc) ---')
eval_traj(model, clip_items(DATA/'pb_2min'), 'home')
eval_traj(model, temporal_split(clip_items(DATA/'pb_3min_court2')), 'court2')
eval_traj(model, temporal_split(clip_items(DATA/'pb_3min_indoor')), 'indoor')
